---
# RL02: Deep RL with Policy Gradient methods.
---

Welcome to the second part of the RL Lab @ M2L 2024!

This notebook introduces you to a fundamental concept in reinforcement learning: **Policy Gradient Methods**. Why should we care? Many real-world problems involve continuous action spaces or complex environments where value-based methods struggle to succeed. Policy gradient methods offer an alternative approach by directly learning a policy, making them powerful tools for these kinds of challenges.

Our goal in this notebook is to take you step-by-step through the process of building and training an agent using policy gradient methods and neural networks. By the end, you'll be able to implement a neural network that learns a policy to solve classic control tasks. Here’s what this journey will involve:

- **Exploring Policy Gradients**: We'll introduce you to the core ideas behind policy gradient methods and why they're a natural fit for problems where traditional RL methods fall short.
- **Neural Network as Policy Approximators**: Learn how deep neural networks can be used to approximate the policy, enabling agents to navigate complex environments.
- **Implementing the REINFORCE (and more) Algorithm**: You'll code the REINFORCE algorithm, one of the most widely used policy gradient methods.
- **Training an Agent in a Classic Control Environment**: Finally, we'll put everything together and train an agent to master a classic control environment using your own implementation.

The exercises and explanations provided here are designed to merge theoretical understanding with practical coding. By the time you finish, you'll have hands-on experience with policy gradient methods and their applications in reinforcement learning.



In [ ]:
# Install dependecies
!pip install gymnasium

In [ ]:
# Imports
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib import rc
import seaborn as sns
from tqdm import tqdm
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.functional import one_hot
import torch.optim as optim
from torch.distributions import Categorical
import gym

# making plots pretty
sns.set_palette("deep")
rc('animation', html='jshtml')
import warnings
warnings.filterwarnings('ignore')

# for reproducibility's sake!
random.seed(42)
torch.manual_seed(42)
print(gym.__version__)

In [ ]:
#@title Some helper functions

def evaluate(env, policy, gamma=1., num_episodes=100):
    """
    Evaluate a RL agent
    :param env: (Env object) the Gym environment
    :param policy: (BasePolicy object) the policy in stable_baselines3
    :param gamma: (float) the discount factor
    :param num_episodes: (int) number of episodes to evaluate it
    :return: (float) Mean reward for the last num_episodes
    """
    all_episode_rewards = []
    for i in range(num_episodes): # iterate over the episodes
        episode_rewards = []
        done = False
        discounter = 1.
        obs = env.reset()
        frames = []
        while not done: # iterate over the steps until termination
            action = policy.draw_action(obs)
            obs, reward, terminated, truncated = env.step(action)
            done = terminated or truncated
            episode_rewards.append(reward * discounter) # compute discounted reward
            discounter *= gamma
            frames.append(env.render(mode="rgb_array"))

        all_episode_rewards.append(sum(episode_rewards))

    mean_episode_reward = np.mean(all_episode_rewards)
    std_episode_reward = np.std(all_episode_rewards) / np.sqrt(num_episodes - 1)
    print("Mean reward:", mean_episode_reward,
          "Std reward:", std_episode_reward,
          "Num episodes:", num_episodes)

    return mean_episode_reward, std_episode_reward, frames



def plot_results(results):
    """
    Plot the results of the experiments
    :param results: (list of tuples) each tuple contains the mean and std of the return
    """

    plt.figure()

    _mean = []
    _std = []
    for m, s, _ in results:
        _mean.append(m)
        _std.append(s)

    _mean = np.array(_mean)
    _std = np.array(_std)

    ts = np.arange(len(_mean))
    plt.plot(ts, _mean)
    plt.fill_between(ts, _mean-_std, _mean+_std, alpha=.2)

    plt.xlabel('Trajectories')
    plt.ylabel('Average return')
    plt.legend(loc='lower right')

    plt.show()


def collect_rollouts(env, policy, m, T):
    """
    Collects m rollouts by running the policy in the
        environment
    :param env: (Env object) the Gym environment
    :param policy: (Policy object) the policy
    :param gamma: (float) the discount factor
    :param m: (int) number of episodes per iterations
    :param K: (int) maximum number of iterations
    :param theta0: (ndarray) initial parameters (d,)
    :param alpha: (float) the constant learning rate
    :param T: (int) the trajectory horizon
    :return: (list of lists) one list per episode
                each containing triples (s, a, r)
    """

    ll = []
    for j in range(m):
        s = env.reset()
        t = 0
        done = False
        l = []
        while t < T and not done:
            a = policy.draw_action(s)
            s1, r, done, _ = env.step(a)
            l.append((s, a, r))
            s = s1
            t += 1
        ll.append(l)
    return ll

def animate(data, interval=200):
  fig = plt.figure(1)
  img = plt.imshow(data[0][0])
  plt.axis('off')

  def animate(i):
    img.set_data(data[i][0])

  anim = animation.FuncAnimation(fig, animate, frames=len(data), interval=interval)
  plt.close(1)
  return anim

# Introduction to Policy Gradient RL

In this section, we are going to look at alternative to **value-based methods** approach: **policy gradient methods**.
The name "policy gradient" comes from the fact that we are estimating the gradient of the policy (rather than Q-function). **Policy-based methods** use iterative update rules to calculate the expected return associated with a state and action.

In order to learn, we need a loss function or *objective*. In RL, the general objective is to maximise the expected episode return by taking actions in the environment. Suppose that the agent's policy is parametrised by a function with paramethers $\theta$: then the actions are determined by $\pi_\theta(a|s)$. A very general way to represent a policy is with a neural network with parameters $\theta$. So, the task of RL is to find the neural network parameters $\theta$ that maximise

$$J(\pi_\theta)=\mathrm{E}_{\tau\sim\pi_\theta}\ [R(\tau)],$$

where $\mathrm{E}$ means *expectation*, $\tau$ is again shorthand for episode, and $R(\tau)$ denotes the return of episode $\tau$.

Then, the goal in RL is to find the parameters $\theta$ that maximise the function $J(\pi_\theta)$. One way to find the parameters $\theta$ that maximise $J(\pi_\theta)$ is to perform gradient ascent on $J(\pi_\theta)$ with respect to the parameters $\theta$.

$$\theta_{k+1}=\theta_k + \alpha \nabla J(\pi_\theta)|_{\theta_{k}},$$

where $\nabla J(\pi_\theta)|_{\theta_{k}}$ is the gradient of the expected return with respect to the policy parameters $\theta_k$ and $\alpha$ is the step size. This quantity, $\nabla J(\pi_\theta)$, is also called the **policy gradient**. If we can compute the policy gradient, then we will have a means by which to directly optimise our policy.

As it turns out, there is a [way](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html) for us to compute the policy gradient:


$$\nabla_{\theta} J(\pi_{\theta})=\underset{\tau \sim \pi_{\theta}}{\mathrm{E}}[\sum_{t=0}^{T} \nabla_{\theta} \log \pi_{\theta}(a_{t} \mid s_{t}) R(\tau)].$$

Informaly, the policy gradient is equal to the gradient of the log of the probability of the action chosen, multiplied by the return of the episode in which the action was taken.
**REINFORCE** is a simple RL algorithm that uses the policy gradient to find the optimal policy by increasing the probability of choosing actions (*reinforcing* actions) that tend to lead to high return episodes.

To test and understand policy gradient we will once again use the [CartPole](https://gymnasium.farama.org/environments/classic_control/cart_pole/) or inverted pendulum environment.

*Recalls from the first part*:
- An **environment** represent the task or the problem that we are trying to solve. Our agent directly interacts with the environment through an **action** collecting a **reward**, which will drive the learning process, and observing the variation of the **state**.
- An **episode** (also called **trajectory** or **rollout**) refers to the sequence of states, actions and rewards an agent experiences as it interacts with the environment over time.

In [ ]:
env_id = "CartPole-v1"

# Create the env
env = gym.make(env_id, render_mode="rgb_array")

# Create the evaluation env
eval_env = gym.make(env_id, render_mode="rgb_array")

# Get the state space and action space
s_size = env.observation_space.shape[0]
a_size = env.action_space.n

In [ ]:
print("Observation space \n")
print("The State Space is: ", s_size)
print("Sample observation", env.observation_space.sample()) # Get a random observation

In [ ]:
print("Action space \n")
print("The Action Space is: ", a_size)
print("Action Space Sample", env.action_space.sample()) # Take a random action

## Policy

In reinforcement learning (RL), a **policy** defines the behavior of an agent by mapping states to actions. It determines how the agent makes decisions at each time step while interacting with the environment. Formally, a policy $\pi$ is a function that takes a state as inpunt and returns an action.
We distinguish two types of policies:
- **Deterministic Policy**: Always returns the same action for a given state.

$$
  a = \pi(s)
$$

- **Stochastic Policy**: Returns a probability distribution over possible actions. The agent samples an action from this distribution.

$$
  \pi(a | s) = P(a | s)
$$

In stochastic policies, $\pi(a∣s)$ represents the probability of taking action $a$ given the state $s$.

More sophisticated policies lead to more complex behavior for our agent. In complex environment with high-dimensional state action spaces it might be useful to use neural networks to represent our policy. The **policy network** takes the current state of the environment as input and outputs either a specific action (for deterministic policies) or a probability distribution over possible actions (for stochastic policies). The network’s weights are learned through training, enabling the agent to make decisions that maximize cumulative rewards over time.

For our environment we will use a **softmax policy**. A softmax policy is a type of stochastic policy where the agent selects actions based on probabilities derived from a softmax function. The probability of taking an action $a$ is calculated as:

$$
  \pi(a|s) = \frac{\exp(h_\theta(s,a))}{\sum_{a^\prime}\exp(h_\theta(s,a^\prime))}
$$

Intuitively, the probability of an action is weighted by its *energy*, or *preference* $h_\theta(s,a)$. We can then compute the score of a softmax policy as:


$$
\begin{aligned}
\nabla_{\boldsymbol{\theta}} \log \pi_{\boldsymbol{\theta}}(a \mid s) & =\nabla_{\boldsymbol{\theta}} h_{\boldsymbol{\theta}}(s, a)-\nabla_{\boldsymbol{\theta}} \log \sum_{a^{\prime} \in \mathcal{A}} \exp \left(h_{\boldsymbol{\theta}}\left(s, a^{\prime}\right) \right) \\
& =\nabla_{\boldsymbol{\theta}} h_{\boldsymbol{\theta}}(s, a)-\frac{\sum_{a^{\prime} \in \mathcal{A}} \nabla_{\boldsymbol{\theta}} \exp \left(h_{\boldsymbol{\theta}}\left(s, a^{\prime}\right)\right)}{\sum_{a^{\prime} \in \mathcal{A}} \exp \left(h_{\boldsymbol{\theta}}\left(s, a^{\prime}\right)\right)} \\
& =\nabla_{\boldsymbol{\theta}} h_{\boldsymbol{\theta}}(s, a)-\sum_{a^{\prime} \in \mathcal{A}} \frac{\exp \left(h_{\boldsymbol{\theta}}\left(s, a^{\prime}\right)\right)}{\sum_{a^{\prime \prime} \in \mathcal{A}} \exp \left(h_{\boldsymbol{\theta}}\left(s, a^{\prime \prime}\right)\right)} \nabla_{\boldsymbol{\theta}} h_{\boldsymbol{\theta}}\left(s, a^{\prime}\right) \\
& =\left(\nabla_{\boldsymbol{\theta}} h_{\boldsymbol{\theta}}(s, a)-\underset{a^{\prime} \sim \pi_{\boldsymbol{\theta}}(\cdot \mid s)}{\mathbb{E}}\left[\nabla_{\boldsymbol{\theta}} h_{\boldsymbol{\theta}}\left(s, a^{\prime}\right)\right]\right)
\end{aligned}
$$


In the next cell we will implement a policy network with a softmax output.

## ⭐ Exercise

1) **Policy neural network.**
Our policy neural network policy will take the observation as input and passes it through an MLP with `len(num_hiddens)` hidden layers and then outputs one scalar value for each of the possible actions (`2` in CartPole). The outputs of our policy network are the action probabilities provided by a softmax function. Have a look [here](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html) to understand how to use a softmax using PyTorch. We will use PyTorch for the neural network training. Please have a look [here](https://pytorch.org/docs/stable/index.html) to understand how PyTorch implements neural network layers and activation functions.

2) **Actor function.**
Next we implement `draw_action` function, which takes network parameters, timestep and random key and returns an action of the policy. Fill in the gaps in the `actor_step` function. Use [`torch.distributions.categorical`](https://pytorch.org/docs/stable/distributions.html#categorical) function to sample an action given the logits.

3) **Compute the gradient log.**
Finally, we implement the function that computes the score function (or gradient log) of the parameters, which we will later use to estimate the policy gradient.


In [ ]:
class SoftmaxPolicyNetwork(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=16):
        super(SoftmaxPolicyNetwork, self).__init__()
        # -----------------------------------#
        # Define the neural network linear layers
        # using nn and the defined dims (in, hidden, out)
        self.fc1 =
        self.fc2 =
        # -----------------------------------#
        self.dim = self.get_parameters().size(dim=0)

    def forward(self, state):
        # -----------------------------------#
        # Forward pass through the network
        # fc1 -> relu -> fc2, using nn and F modules
        x =
        logits =
        # -----------------------------------#
        action_probs = F.softmax(logits, dim=-1)
        return action_probs

    def get_parameters(self):
        return nn.utils.parameters_to_vector(self.parameters()).detach()

    def set_parameters(self, new_params):
        nn.utils.vector_to_parameters(new_params, self.parameters())

    def draw_action(self, state):
        # -----------------------------------#
        # Select an action based on the state using the softmax policy.
        # use forward() and sample from action_probs
        state = torch.FloatTensor(state)
        action_probs =
        action = Categorical('''fill here''').sample().item()
        # -----------------------------------#
        return action

    def grad_log(self, state, action):
        """
        Compute the score function (policy gradient) for a given state and action.

        Args:
            state: A tensor representing the current state (1D).
            action: The action taken (int).

        Returns:
            score: A 1D tensor representing the gradient of the log-probability
                   of the action with respect to the network parameters.
        """
        state = torch.FloatTensor(state)
        action_probs = self.forward(state) # soft

        # -----------------------------------#
        # One-hot encoding of the selected action
        # we convert (soft) action probs into a (hard) action one-hot idx
        action_one_hot =
        action_one_hot[action] =

        # Compute the gradient of the log-probability w.r.t logits
        # that is the difference between hard and soft probs
        score =
        # -----------------------------------#

        # Compute the gradient of the network parameters using chain rule
        # First, we propagate the score through the second layer (fc2)
        fc2_grad_wrt_logits = score.view(-1, 1)  # score is d(log P(a|s)) / d(logits)
        fc2_grad_wrt_weights = torch.matmul(fc2_grad_wrt_logits, F.relu(self.fc1(state)).view(1, -1))  # Chain rule for fc2 weights
        fc2_grad_wrt_bias = fc2_grad_wrt_logits.view(-1)  # Chain rule for fc2 bias

        # Now, we compute the gradient w.r.t. the output of the first layer
        d_fc1_output = torch.matmul(self.fc2.weight.T, score) * (self.fc1(state) > 0).float()  # Backprop through ReLU
        fc1_grad_wrt_weights = torch.matmul(d_fc1_output.view(-1, 1), state.view(1, -1))  # Chain rule for fc1 weights
        fc1_grad_wrt_bias = d_fc1_output.view(-1)  # Chain rule for fc1 bias

        # Concatenate all gradients into a single vector
        score_vector = torch.cat([fc1_grad_wrt_weights.view(-1), fc1_grad_wrt_bias.view(-1),
                                  fc2_grad_wrt_weights.view(-1), fc2_grad_wrt_bias.view(-1)])

        return score_vector

# Policy gradient algorithm

In this section, we will explore two widely used policy gradient algorithms: **REINFORCE** and **GPOMDP**. Both methods aim to optimize policies in reinforcement learning by estimating the gradient of the expected reward with respect to policy parameters. First, we provide the mathematical formulations of the gradient estimators for each algorithm, highlighting their derivations and key differences. Following this, we provide their implementation in Python.

## REINFORCE

[REINFORCE](https://link.springer.com/content/pdf/10.1007/BF00992696.pdf) (Monte-Carlo policy gradient) relies on an estimated return by Monte-Carlo methods using episode samples to update the policy parameter $\theta$. REINFORCE works because the expectation of the sample gradient is equal to the actual gradient:

$$
\begin{aligned}
\nabla_\theta J(\theta) & =\mathbb{E}_\pi\left[Q^\pi(s, a) \nabla_\theta \log \pi_\theta(a \mid s)\right] \\
& =\mathbb{E}_\pi\left[G_t \nabla_\theta \log \pi_\theta\left(A_t \mid S_t\right)\right]
\end{aligned}
$$

Therefore we are able to measure  from real sample trajectories and use that to update our policy gradient. It relies on a full trajectory and that’s why it is a Monte-Carlo method. We can define the gradient estimator as:

$$
\begin{aligned} \nabla_{\boldsymbol{\theta}} J(\boldsymbol{\theta}) & =\mathbb{E}_{\tau \sim p_{\boldsymbol{\theta}}}\left[\nabla_{\boldsymbol{\theta}} \log p_{\boldsymbol{\theta}}(\tau) G(\tau)\right] \\ & =\mathbb{E}_{\tau \sim p_{\boldsymbol{\theta}}}\left[\left(\sum_{l=0}^{T-1} \nabla_{\boldsymbol{\theta}} \log \pi_{\boldsymbol{\theta}}\left(A_l \mid S_l\right)\right)\left(\sum_{t=0}^{T-1} \gamma^t r\left(S_t, A_t\right)\right)\right] \\
& = \frac{1}{N} \sum_{i=1}^N \nabla_\theta \log \pi_\theta(a_{i} | s_{i})R_i
\end{aligned}
$$

The process is pretty straightforward:

- Initialize the policy parameter $\theta$.
- Generate one trajectory on policy $\pi_\theta$: .
- For t=1, 2, … , T:
  - Estimate the the return $R$;
  - Update policy parameters: $\theta \leftarrow \theta+\alpha \widehat{\nabla}_{\theta}^{\text{REINFORCE}} J(\theta)$

### ⭐ Exercise

In the next cell implement the REINFORCE estimator given the formal definition presented in the previous section.



In [ ]:
def reinforce(rollouts, policy, gamma):

    grad = 0

    for roll in rollouts:
        H = len(roll)
        disc_rew = torch.zeros((H, 1))
        scores = torch.zeros((H, policy.dim))

        # Retrieve the score of the policy and compute the gradient
        for t in range(H):
            s, a, r = roll[t] # takes one timestep
            disc_rew[t] = gamma ** t * r # new discount
            scores[t] = policy.grad_log(s, a)

        # -----------------------------------#
        # multiply the new discounts tensor with the scores and
        # sum over the horizon H (first dimension)
        grad +=
        # -----------------------------------#

    return grad/len(rollouts)

## GPOMDP

Performing gradient ascent on the gradient of the log of the action probability, weighted by the return of the episode will tend to push up the probability of actions that were in episodes with high return, regardless of *where* in the episode the action was taken. This does not really make much sense because an action near the end of an episode may be reinforced because lots of reward was collected earlier on in the episode, *before* the action was taken. RL agents should really only reinforce actions on the basis of their *consequences*. Rewards obtained before taking an action have no bearing on how good that action was: only rewards that come after. The cummulative rewards received after an action was taken is called the **rewards-to-go** and can be computed as:

$$\hat{R}_i=\sum_{t=i}^Tr_t.$$

We can improve the reliability of the policy gradient by substituting the episode return with the rewards-to-go. The [GPOMDP](https://arxiv.org/pdf/1106.0665) gradient estimator exploits the rewards-to-go to provide an estimation more reliable and with lower variance w.r.t. the REINFORCE implementation. The gradient estimator is given by:

$$
\begin{aligned}
    \nabla_{\theta} J({\theta}) & = \frac{1}{N} \sum_{i=1}^N \sum_{t=0}^{H-1} \gamma^t r(s_{i,t},a_{i,t}) \sum_{l=0}^t \nabla_\theta \log \pi_\theta(a_{i,l} | s_{i,l}) \\
    & = \frac{1}{N} \sum_{i=1}^N \sum_{l=0}^t \nabla_\theta \log \pi_\theta(a_{i,l} | s_{i,l})\hat{R}_i
\end{aligned}
$$

The process is similar to the REINFORCE algorithm:
The process is pretty straightforward:

- Initialize the policy parameter $\theta$.
- Generate one trajectory on policy $\pi_\theta$: .
- For t=1, 2, … , T:
  - Estimate the return $\hat{R}$;
  - Update policy parameters: $\theta \leftarrow \theta+\alpha \widehat{\nabla}_{\theta}^{\text{GPOMDP}} J(\theta)$

### ⭐ Exercise

In the next cell implement the GPOMDP estimator given the formal definition presented in the previous section.

Hint: take a look at [`torch.cumsum`](https://pytorch.org/docs/stable/generated/torch.cumsum.html) for the GPOMDP implementation.

In [ ]:
def gpomdp(rollouts, policy, gamma):

    grad = 0

    for roll in rollouts:
        H = len(roll)
        disc_rew = torch.zeros((H, 1))
        scores = torch.zeros((H, policy.dim))

        # Retrive the score of the policy and compute the gradient
        for t in range(H):
            s, a, r = roll[t]
            disc_rew[t] = gamma ** t * r
            scores[t] = policy.grad_log(s, a)

        # -----------------------------------#
        # We want to compute the cumulative sum of scores along the first dim
        # and again, multiply it by the discounts and sum
        cum_scores =
        grad +=
        # -----------------------------------#

    return grad / len(rollouts)

# Training loop

Finally, let's run the learning loop.
As the structure of the agent is very similar to Q-learning agent, the learning loop is very similar as well. Notice that we perform a learning step on a batch of samples instead of a single datapoint.

In [ ]:

def train(env, policy, gamma, m, K, alpha, T, estimator):
    """
    Train a policy with G(PO)MDP
    :param env: (Env object) the Gym environment
    :param policy: (Policy object) the policy
    :param gamma: (float) the discount factor
    :param m: (int) number of episodes per iterations
    :param K: (int) maximum number of iterations
    :param alpha: (float) the constant learning rate
    :param T: (int) the trajectory horizon
    :return: list (ndarray, ndarray) the evaluations
    """

    results = []

    # Evaluate the initial policy
    res = evaluate(env, policy, gamma)
    results.append(res)

    for k in range(K):

        print('Iteration:', k)

        # Generate rollouts
        rollouts = collect_rollouts(env, policy, m, T)

        # Get policy parameter
        theta = policy.get_parameters()

        # Call your Gradient estimator
        if estimator == 'REINFORCE':
          pg = reinforce(rollouts, policy, gamma)
        else:
          pg = gpomdp(rollouts, policy, gamma)

        # Update policy parameter
        theta = theta + alpha * pg

        # Set policy parameters
        policy.set_parameters(theta)

        # Evaluate the updated policy
        res = evaluate(env, policy, gamma)
        results.append(res)

    return results

## ⭐ Exercise

- Choose the parameters (number of training episodes, numbers of hidden units in MLP, learning rate, batch size) so that the average return of the agent increases with training and ends up being greater than $100$.
- Run the learning loop and visualise the epsiode returns. Look at the animation of the last episode. Optional: modify the code and try both gradient estimators. What can you notice?

In [ ]:
discount_factor = 0.999 # @param
batch_size = 100
iterations = 100 # @param
learning_rate = 0.001 # @param
trajectory_lenght = 200 # @param
n_hidden = 8 # @param

# gradient estimator {'REINFORCE', 'gpomdp'}
estimator = 'gpomdp' # @param

# Instantiate the policy
policy = SoftmaxPolicyNetwork(s_size, a_size, n_hidden)

# Start training
results = train(env, policy, discount_factor, batch_size, iterations, learning_rate, trajectory_lenght, estimator)

In [ ]:
# Plot the results of the training
plot_results(results)

In [ ]:
# Evaluate the policy and animate the results
perf_mean, perf_std, frames = evaluate(eval_env, policy, num_episodes=1)

# Animate the last episode
animate(frames)

# 🥳 Congratulations on completing the second part of this tutorial, great job!!!

In this part you learnt how policy gradient works and how to combine it with neural networks. Next, we will look into RL from Human Feedback (RLHF).

---
# Bonus: PPO on a Continuous Control Task

## Step 1 — From REINFORCE to PPO

You have just implemented REINFORCE and GPOMDP by hand. These algorithms are the theoretical foundation of modern policy gradient research. Now let's see what happens when we apply these ideas to a **much harder task**, using a **state-of-the-art algorithm**.

### The core problem with vanilla policy gradient

Vanilla policy gradient methods (REINFORCE, GPOMDP) suffer from a critical practical issue: it is very hard to choose the right step size $\alpha$.

- **Too large**: a single bad update sends the policy to a region of parameter space from which it can never recover (catastrophic forgetting).
- **Too small**: learning is agonisingly slow.

The root cause is that a small change in the parameters $\theta$ can produce a *large* change in the policy $\pi_\theta$ — especially when using neural networks. Classical gradient ascent gives no protection against these destructive updates.

### PPO: the Proximal Policy Optimization solution

**PPO** [(Schulman et al., 2017)](https://arxiv.org/abs/1707.06347) solves this with a **clipped surrogate objective** that *mathematically constrains* how much the policy can change in a single update.

Let $r_t(\theta) = \dfrac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{\text{old}}}(a_t | s_t)}$ be the **probability ratio** between the new and old policy. If $r_t > 1$, the new policy assigns *more* probability to the action; if $r_t < 1$, *less*.

The PPO objective clips this ratio to stay inside $[1 - \varepsilon,\ 1 + \varepsilon]$:

$$
L^{\text{CLIP}}(\theta) = \mathbb{E}_t \Bigl[\min\bigl(\underbrace{r_t(\theta)\,\hat{A}_t}_{\text{standard PG}},\ \underbrace{\operatorname{clip}(r_t(\theta),\, 1{-}\varepsilon,\, 1{+}\varepsilon)\,\hat{A}_t}_{\text{conservative bound}}\bigr)\Bigr]
$$

where $\hat{A}_t$ is the **estimated advantage** — how much better (or worse) than average this action turned out to be. The $\min$ ensures we take the *pessimistic* bound: we never gain from pushing the ratio past the clip boundary. In practice $\varepsilon = 0.2$.

### Advantage estimation with GAE

Instead of using the raw return (like REINFORCE) or rewards-to-go (like GPOMDP), PPO uses **Generalised Advantage Estimation (GAE)**:

$$
\hat{A}_t = \sum_{l=0}^{\infty} (\gamma \lambda)^l \,\delta_{t+l}, \qquad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
$$

where $V(s)$ is a learned **value function** (the critic) and $\lambda \in [0,1]$ controls the bias-variance trade-off. This is the generalisation of rewards-to-go to the actor-critic setting.

### How PPO compares to what you built

| Feature | REINFORCE / GPOMDP | PPO |
|---|---|---|
| Update stability | No constraint on step size | ✓ Clipped surrogate objective |
| Variance reduction | None (or rewards-to-go) | ✓ GAE with a learned value baseline |
| Architecture | Policy network only | Actor-Critic (policy + value heads) |
| Sample efficiency | 1 gradient step per batch | Multiple gradient epochs per batch |
| Action space | Discrete (softmax) | Both discrete **and continuous** (Gaussian) |

PPO has become one of the default baselines in deep RL research and is used in production systems including early RLHF pipelines for large language models.

In [ ]:
# stable-baselines3 — RL algorithm library
# mujoco — physics simulator required by HalfCheetah-v4
!pip install stable-baselines3[extra] mujoco --quiet

## Step 2 — The HalfCheetah environment

[HalfCheetah-v4](https://gymnasium.farama.org/environments/mujoco/half_cheetah/) is a classic continuous control benchmark from the [MuJoCo](https://mujoco.org/) physics simulator. The agent must learn to make a 2D planar cheetah robot run forward as fast as possible by controlling the torques applied to its 6 joints.

### Why this is harder than CartPole

| | CartPole-v1 | HalfCheetah-v4 |
|---|---|---|
| State dim | 4 | **17** (joint angles + velocities) |
| Action space | Discrete {left, right} | **Continuous** $[-1,1]^6$ (6 joint torques) |
| Policy type | Softmax (categorical) | **Gaussian** $\mathcal{N}(\mu_\theta(s), \sigma_\theta(s))$ |
| Reward | +1 per timestep balanced | forward speed − control cost |
| Max return | 500 | ~3 000+ |
| Steps to solve | ~50 k | **~1–3 M** |

### From softmax to Gaussian policy

The softmax policy you implemented picks a discrete action index. It cannot generalise to continuous actions. For continuous spaces, PPO instead learns a **Gaussian policy**:

$$\pi_\theta(a \mid s) = \prod_{i=1}^{6} \mathcal{N}\!\left(a_i \;\big|\; \mu_\theta^{(i)}(s),\; \sigma_\theta^{(i)}(s)\right)$$

The neural network outputs a **mean** $\mu_\theta(s) \in \mathbb{R}^6$ for each joint and a learnable **log-standard-deviation** $\log\sigma$. An action is sampled from this factored Gaussian and clipped to $[-1,1]$. The policy gradient formula is the same — we just need $\nabla_\theta \log \pi_\theta(a|s)$, which has a clean closed form for Gaussians.

Stable Baselines3 handles all of this automatically.

Let's inspect the environment spaces first:

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

sns.set_palette("deep")

# Create the environment just to inspect its spaces
_env = gym.make("HalfCheetah-v4")

print("=== HalfCheetah-v4 ===\n")
print(f"Observation space : {_env.observation_space}")
print(f"  shape           : {_env.observation_space.shape}   (17 joint angles & velocities)")
print()
print(f"Action space      : {_env.action_space}")
print(f"  shape           : {_env.action_space.shape}    (6 joint torques)")
print(f"  bounds          : [{_env.action_space.low[0]:.1f}, {_env.action_space.high[0]:.1f}]  per dimension")
print()
print("Reward function   : forward_velocity - 0.1 * control_cost")

_env.close()

## Step 3 — Setting up PPO with Stable Baselines3

### What is Stable Baselines3?

[Stable Baselines3 (SB3)](https://stable-baselines3.readthedocs.io/) is a set of reliable, well-tested PyTorch implementations of popular deep RL algorithms (PPO, SAC, TD3, DQN, …). Rather than re-implementing PPO from scratch (which is notoriously tricky to get right, with many subtle implementation details that affect performance), we use SB3 so we can focus on *understanding* the algorithm.

### The Actor-Critic network SB3 builds for you

When you pass `policy="MlpPolicy"` to a continuous-action environment, SB3 automatically creates the following architecture:

```
Observation (17,)
      │
┌─────▼──────────────────────────┐
│  Shared trunk (optional)       │
│  (by default: no shared layers)│
└──────────┬───────────┬─────────┘
           │           │
  ┌────────▼───┐  ┌────▼────────┐
  │   Actor    │  │   Critic    │
  │ Linear→64  │  │ Linear→64  │
  │   Tanh     │  │   Tanh     │
  │ Linear→64  │  │ Linear→64  │
  │   Tanh     │  │   Tanh     │
  │ Linear→6   │  │ Linear→1   │
  │ (mean μ)   │  │  V(s) ──── used to compute Â_t
  │ + log σ    │  └────────────┘
  └────────────┘
```

- **Actor head**: outputs $\mu_\theta(s)$ (mean) and a separate learnable $\log\sigma$ for each action dimension. An action is sampled as $a \sim \mathcal{N}(\mu_\theta(s), \sigma_\theta(s))$.
- **Critic head**: outputs $V_\phi(s)$, a scalar estimate of the expected return from state $s$. This is used as a **baseline** to reduce the variance of the gradient estimates — exactly the idea behind rewards-to-go, but now using a *learned* baseline.

### Key hyper-parameters

| Parameter | Value | What it controls |
|---|---|---|
| `n_steps` | 2048 | Steps collected per env before each update. Larger = lower-variance gradient but slower wall-clock. |
| `batch_size` | 64 | SGD mini-batch size. Must divide `n_steps`. |
| `n_epochs` | 10 | Gradient passes over each collected batch. More = better sample efficiency, but risks overfitting stale data. |
| `clip_range` ε | 0.2 | Max allowed change in the probability ratio $r_t$. Smaller = more conservative updates. |
| `gae_lambda` λ | 0.95 | GAE smoothing between Monte-Carlo (λ=1, high variance) and TD(0) (λ=0, high bias). |
| `gamma` γ | 0.99 | Discount factor — how much future rewards are worth. |
| `learning_rate` | 3e-4 | Adam step size for both actor and critic optimisers. |

In [ ]:
train_env = gym.make("HalfCheetah-v4")
eval_env  = Monitor(gym.make("HalfCheetah-v4"))

model = PPO(
    policy        = "MlpPolicy", # actor-critic MLP, auto-detects continuous action space
    env           = train_env,
    # ── optimiser ──────────────────────────────────────────
    learning_rate = 3e-4,        # Adam step size for both actor and critic
    # ── data collection ────────────────────────────────────
    n_steps       = 2048,        # environment steps collected before each update
    batch_size    = 64,          # SGD mini-batch size (must divide n_steps)
    # ── update ─────────────────────────────────────────────
    n_epochs      = 10,          # gradient passes over each collected batch
    clip_range    = 0.2,         # ε — the clipping threshold in L^CLIP
    # ── returns & advantages ────────────────────────────────
    gamma         = 0.99,        # discount factor γ
    gae_lambda    = 0.95,        # GAE λ: 1.0 = Monte-Carlo, 0.0 = TD(0)
    # ── misc ────────────────────────────────────────────────
    ent_coef      = 0.0,         # entropy bonus (0 = no explicit exploration term)
    verbose       = 0,
    seed          = 42,
)

# Inspect the network SB3 built for us
print(model.policy)

## Step 4 — Training the agent

We will train for **500 000 environment steps**, evaluating every 50 000 steps to track progress.

### What `model.learn()` does under the hood

Each time we call `model.learn(total_timesteps=N)`, SB3 repeats the following loop until `N` steps have been collected:

```
┌─ Collect ────────────────────────────────────────────────────────────┐
│  Run current policy π_θ in the environment for n_steps = 2 048 steps │
│  Store (s, a, r, s', done) transitions in a rollout buffer           │
└──────────────────────────────────────────────────────────────────────┘
         │
         ▼
┌─ Compute advantages ─────────────────────────────────────────────────┐
│  Use the critic V_φ(s) and GAE (λ=0.95) to compute Â_t for each step │
└──────────────────────────────────────────────────────────────────────┘
         │
         ▼
┌─ Optimise ───────────────────────────────────────────────────────────┐
│  For n_epochs = 10 passes over the rollout buffer (in mini-batches): │
│    • Actor loss  : L^CLIP(θ) — clipped surrogate objective           │
│    • Critic loss : MSE between V_φ(s) and the empirical returns      │
│    • Update θ and φ together with Adam                               │
└──────────────────────────────────────────────────────────────────────┘
         │
         ▼
   Discard rollout buffer  →  repeat with updated policy
```

> **Practical note:** PPO is an *on-policy* algorithm — it can only train on data collected by the current policy. This is why the buffer is discarded after each update. Contrast this with off-policy methods (SAC, TD3) which store all past experience in a replay buffer and reuse it.

> **Expected duration:** ~5–10 minutes on CPU. HalfCheetah-v4 is a fast simulator. You will already see the return increasing within the first 100 000 steps. Expert-level performance (>3 000 reward) typically requires 1–3 M steps.

In [ ]:
TOTAL_TIMESTEPS    = 500_000
N_INTERVALS        = 10
STEPS_PER_INTERVAL = TOTAL_TIMESTEPS // N_INTERVALS   # 50 000 steps between evaluations

mean_rewards, std_rewards, timesteps_log = [], [], []

print(f"Training for {TOTAL_TIMESTEPS:,} steps ({N_INTERVALS} intervals of {STEPS_PER_INTERVAL:,})\n")
print(f"{'Timesteps':>10}  {'Mean return':>12}  {'Std':>8}")
print("─" * 36)

for i in range(N_INTERVALS):
    # model.learn() resumes from where it left off when reset_num_timesteps=False
    model.learn(
        total_timesteps     = STEPS_PER_INTERVAL,
        reset_num_timesteps = (i == 0),   # only reset counter at the very first call
    )

    # evaluate_policy runs the policy deterministically and returns mean ± std
    mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10, warn=False)

    t = (i + 1) * STEPS_PER_INTERVAL
    mean_rewards.append(mean_r)
    std_rewards.append(std_r)
    timesteps_log.append(t)
    print(f"{t:>10,}  {mean_r:>12.1f}  {std_r:>8.1f}")

train_env.close()

In [ ]:
mean_arr = np.array(mean_rewards)
std_arr  = np.array(std_rewards)

plt.figure(figsize=(8, 4))
plt.plot(timesteps_log, mean_arr, marker="o", label="PPO — HalfCheetah-v4")
plt.fill_between(
    timesteps_log,
    mean_arr - std_arr,
    mean_arr + std_arr,
    alpha=0.25,
    label="± 1 std",
)
plt.axhline(3000, color="gray", linestyle="--", linewidth=0.8, label="Expert level (~3 000)")
plt.xlabel("Environment timesteps")
plt.ylabel("Mean episode return (10 episodes)")
plt.title("PPO learning curve — HalfCheetah-v4")
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nFinal performance after {TOTAL_TIMESTEPS:,} steps: {mean_arr[-1]:.1f} ± {std_arr[-1]:.1f}")
print("(A fully-trained agent typically scores > 3 000 after ~2 M steps)")

In [ ]:
# Render one greedy episode with the trained agent and animate it
render_env = gym.make("HalfCheetah-v4", render_mode="rgb_array")
obs, _ = render_env.reset(seed=0)
frames, total_reward, done = [], 0.0, False

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = render_env.step(action)
    done = terminated or truncated
    total_reward += reward
    frames.append(render_env.render())

render_env.close()
print(f"Episode length : {len(frames)} steps")
print(f"Episode return : {total_reward:.1f}")
animate([[f] for f in frames], interval=30)